## Overview
This notebook implements a 3-class exercise classifier (barbell biceps curl, shoulder press, squat) using YOLOv8 pose estimation and a BiLSTM model.

Importing required libraries and installing ultralytics for Yolov8-pose

In [1]:
!pip -q install ultralytics opencv-python
import os, cv2, numpy as np
from ultralytics import YOLO
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import zipfile
import shutil




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Loading Yolov8n-pose

In [3]:
pose_model = YOLO("yolov8n-pose.pt")


# Feature Extraction Functions
Extracting normalized pose keypoints (17 x 3: x, y, confidence) from video frames and creating fixed-length sequences.

In [4]:
SEQ_LEN = 30

def get_pose_kpts_norm(image_bgr):
    """
    Returns (17,3): x_norm, y_norm, conf
    If no person: zeros.
    """
    h, w = image_bgr.shape[:2]
    results = pose_model(image_bgr, verbose=False)
    r = results[0]

    if r.keypoints is None or r.keypoints.xy is None or len(r.keypoints.xy) == 0:
        return np.zeros((17,3), dtype=np.float32)

    xy = r.keypoints.xy[0].cpu().numpy().astype(np.float32)
    conf = r.keypoints.conf[0].cpu().numpy().astype(np.float32)

    xy[:,0] /= max(w, 1)
    xy[:,1] /= max(h, 1)

    kpts = np.concatenate([xy, conf[:,None]], axis=1)
    return kpts

def video_to_sequence_kpts(video_path, seq_len=SEQ_LEN):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        return np.zeros((seq_len,17,3), dtype=np.float32)

    idxs = np.linspace(0, len(frames)-1, seq_len).astype(int)

    seq = []
    for i in idxs:
        kpts = get_pose_kpts_norm(frames[i])
        seq.append(kpts)
    return np.stack(seq, axis=0).astype(np.float32)

Google Drive mount to fetch data

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Loading the json file for kaggle to fetch real-time-exercise dataset

In [7]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\n  "username": "rawaanyasserr",\n  "key": "KGAT_aa69620f6075fdcc5fdc3918c2c3d429"\n}\n'}

Kaggle data setup and extraction

In [8]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [14]:
!kaggle datasets download -d riccardoriccio/real-time-exercise-recognition-dataset -p /content

Dataset URL: https://www.kaggle.com/datasets/riccardoriccio/real-time-exercise-recognition-dataset
License(s): CC-BY-NC-SA-4.0
real-time-exercise-recognition-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [18]:
!mkdir -p /content/data_tmp
!unzip -q /content/real-time-exercise-recognition-dataset.zip -d /content/data_tmp

In [19]:
!ls /content

data_tmp     real-time-exercise-recognition-dataset.zip  yolov8n-pose.pt
drive	     sample_data
kaggle.json  tmp_fitness_aqa_extract


In [20]:
!ls /content/data_tmp

final_kaggle_with_additional_video  my_test_video_1  synthetic_dataset
fitness_aqa_subset		    similar_dataset


In [21]:
!find /content/data_tmp/final_kaggle_with_additional_video -maxdepth 2 -type d

/content/data_tmp/final_kaggle_with_additional_video
/content/data_tmp/final_kaggle_with_additional_video/barbell biceps curl
/content/data_tmp/final_kaggle_with_additional_video/squat
/content/data_tmp/final_kaggle_with_additional_video/hammer curl
/content/data_tmp/final_kaggle_with_additional_video/push-up
/content/data_tmp/final_kaggle_with_additional_video/shoulder press


Creating a subset for the squat data to match the curl and press data size

In [22]:
zip_path = "/content/drive/MyDrive/Fitness-AQA_dataset_release/Squat/Labeled_Dataset/videos.zip"
subset_root = "/content/data_tmp/fitness_aqa_subset"
subset_dir = os.path.join(subset_root, "squat")

if os.path.exists(subset_dir):
    shutil.rmtree(subset_dir)
os.makedirs(subset_dir, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    all_mp4 = [n for n in z.namelist() if n.endswith(".mp4")]

print("Total squat videos in zip:", len(all_mp4))

random.seed(42)
sample_count = 30
sample_videos = random.sample(all_mp4, sample_count)

with zipfile.ZipFile(zip_path) as z:
    for member in sample_videos:
        extracted_path = z.extract(member, "/content/tmp_fitness_aqa_extract")
        filename = os.path.basename(member)
        shutil.move(extracted_path, os.path.join(subset_dir, filename))

print("Subset extracted to:", subset_dir)
print("Videos in subset:", len(os.listdir(subset_dir)))

Total squat videos in zip: 1739
Subset extracted to: /content/data_tmp/fitness_aqa_subset/squat
Videos in subset: 30


# Load Exercise Videos and Extract Pose Keypoints
#
# Dataset Sources:
1. final_kaggle_with_additional_video - Main Kaggle dataset (25 videos per class)
2. similar_dataset - Additional similar videos (18 videos per class)
3. my_exercise_videos - Custom recorded videos (27-32 videos)
4. fitness_aqa_subset - Squat videos from Fitness-AQA (30 videos)

# Parameters:
- SEQ_LEN = 30: Number of frames sampled per video
- pose_model: YOLOv8n-pose for extracting 17 body keypoints
- CLASS_MAP: Maps exercise names to numeric labels

# Output:
- X: Array of pose keypoints (samples × 30 frames × 17 keypoints × 3 features)
- y: Array of labels (0=curl, 1=press, 2=squat)

In [23]:
SEQ_LEN = 30
pose_model = YOLO("yolov8n-pose.pt")

BASES = [
    "/content/data_tmp/final_kaggle_with_additional_video",
    "/content/data_tmp/similar_dataset",
    "/content/drive/MyDrive/my_exercise_videos",
    "/content/data_tmp/fitness_aqa_subset",
]

CLASS_MAP = {
    "barbell biceps curl": 0,
    "shoulder press": 1,
    "squat": 2,
}

IDX_TO_NAME = {v: k for k, v in CLASS_MAP.items()}

def list_videos(folder):
    exts = (".mp4", ".avi", ".mov", ".mkv")
    if not os.path.exists(folder):
        return []
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(exts)
    ])

X, y, base_id = [], [], []

for bi, base in enumerate(BASES):
    print("\nBASE:", base, "(base_id:", bi, ")")
    for class_name, label in CLASS_MAP.items():
        class_dir = os.path.join(base, class_name)
        vids = list_videos(class_dir)
        print(" ", class_name, "->", len(vids), "videos")

        for vp in vids:
            X.append(video_to_sequence_kpts(vp))
            y.append(label)
            base_id.append(bi)

X = np.stack(X, axis=0).astype(np.float32)
y = np.array(y, dtype=np.int64)
base_id = np.array(base_id, dtype=np.int64)

print("\nTOTAL:", len(y))
print("BALANCE:", np.bincount(y))


BASE: /content/data_tmp/final_kaggle_with_additional_video (base_id: 0 )
  barbell biceps curl -> 25 videos
  shoulder press -> 25 videos
  squat -> 25 videos

BASE: /content/data_tmp/similar_dataset (base_id: 1 )
  barbell biceps curl -> 18 videos
  shoulder press -> 18 videos
  squat -> 18 videos

BASE: /content/drive/MyDrive/my_exercise_videos (base_id: 2 )
  barbell biceps curl -> 27 videos
  shoulder press -> 32 videos
  squat -> 0 videos

BASE: /content/data_tmp/fitness_aqa_subset (base_id: 3 )
  barbell biceps curl -> 0 videos
  shoulder press -> 0 videos
  squat -> 30 videos

TOTAL: 218
BALANCE: [70 75 73]


#Biomechanical Feature Engineering
Convert raw keypoints to biomechanically meaningful features:
- Hip-centered coordinates
- Shoulder-width normalized positions
- Joint angles (elbows and knees)
- Final feature dimension: 55 (17x3 + 4 angles)

In [24]:
def get_biomechanical_features(kpts):
    hip_center = (kpts[11, :2] + kpts[12, :2]) / 2.0
    rel_coords = kpts.copy()
    rel_coords[:, :2] = kpts[:, :2] - hip_center

    shoulder_width = np.linalg.norm(kpts[5, :2] - kpts[6, :2]) + 1e-6
    rel_coords[:, :2] /= shoulder_width

    def angle(a, b, c):
        ba = a - b
        bc = c - b
        denom = (np.linalg.norm(ba) * np.linalg.norm(bc)) + 1e-6
        cos_a = np.dot(ba, bc) / denom
        cos_a = np.clip(cos_a, -1.0, 1.0)
        return np.degrees(np.arccos(cos_a))

    l_elbow = angle(rel_coords[5, :2], rel_coords[7, :2], rel_coords[9, :2])
    r_elbow = angle(rel_coords[6, :2], rel_coords[8, :2], rel_coords[10, :2])
    l_knee = angle(rel_coords[11, :2], rel_coords[13, :2], rel_coords[15, :2])
    r_knee = angle(rel_coords[12, :2], rel_coords[14, :2], rel_coords[16, :2])

    angles = np.array([l_elbow, r_elbow, l_knee, r_knee], dtype=np.float32) / 180.0
    return np.concatenate([rel_coords.flatten(), angles]).astype(np.float32)

# Dataset and Model Architecture
Define PyTorch Dataset class with temporal augmentation and BiLSTM model for sequence classification.

In [25]:
class KPTDataset(Dataset):
    def __init__(self, X, y, augment=False):
        processed_X = []
        for sequence in X:
            processed_seq = [get_biomechanical_features(frame) for frame in sequence]
            processed_X.append(processed_seq)

        self.X = torch.tensor(np.array(processed_X), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return int(self.y.shape[0])

    def __getitem__(self, i):
        x = self.X[i].clone()
        yy = self.y[i]

        if self.augment:
            if torch.rand(1).item() < 0.30:
                shift = int(torch.randint(-2, 3, (1,)).item())
                x = torch.roll(x, shifts=shift, dims=0)

            x += torch.randn_like(x) * 0.005

        return x, yy

class ExerciseBiLSTM(nn.Module):
    def __init__(self, in_dim=55, hidden=96, num_classes=3, dropout=0.35):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1, :]))

# Training Setup
Helper functions for evaluation, reproducibility seeding, and training loop.

In [26]:
def seed_all(SEED):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def eval_acc(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred == yb).sum().item()
            total += yb.numel()
    return correct / total if total else 0.0

def get_preds(model, loader, device):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            pred = model(xb.to(device)).argmax(1).cpu().numpy()
            P.append(pred)
            Y.append(yb.numpy())
    return np.concatenate(P), np.concatenate(Y)

#Multi-Seed Training
Train model across 5 different seeds to ensure robustness. Select best model based on validation accuracy.
**Results Summary:**
- Best validation accuracy: 100% (Seed 99)
- Test accuracy: 93.94%
- Average test accuracy: 92.73% ± 4.11%

In [27]:
X_np = np.array(X, dtype=np.float32)
y_np = np.array(y, dtype=np.int64)

best_overall = {
    "seed": None,
    "best_val": -1.0,
    "test_acc": -1.0,
    "state": None
}

all_test_accs = []

for SEED in [1, 7, 42, 99, 123]:
    print("\n" + "=" * 60)
    print("SEED:", SEED)
    print("=" * 60)

    seed_all(SEED)

    idx = np.arange(len(y_np))

    tr_idx, temp_idx = train_test_split(
        idx,
        test_size=0.30,
        stratify=y_np,
        random_state=SEED
    )

    va_idx, te_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        stratify=y_np[temp_idx],
        random_state=SEED
    )

    print("Train:", len(tr_idx), "Val:", len(va_idx), "Test:", len(te_idx))
    print("Train bal:", np.bincount(y_np[tr_idx]))
    print("Val bal:  ", np.bincount(y_np[va_idx]))
    print("Test bal: ", np.bincount(y_np[te_idx]))

    BATCH = 32
    train_loader = DataLoader(
        KPTDataset(X_np[tr_idx], y_np[tr_idx], augment=True),
        batch_size=BATCH,
        shuffle=True
    )
    val_loader = DataLoader(
        KPTDataset(X_np[va_idx], y_np[va_idx], augment=False),
        batch_size=BATCH,
        shuffle=False
    )
    test_loader = DataLoader(
        KPTDataset(X_np[te_idx], y_np[te_idx], augment=False),
        batch_size=BATCH,
        shuffle=False
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = ExerciseBiLSTM(
        in_dim=55,
        hidden=96,
        num_classes=3,
        dropout=0.35
    ).to(device)

    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=5e-4
    )

    best_val = 0.0
    best_state = None
    patience = 10
    bad = 0
    EPOCHS = 40

    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        train_acc = eval_acc(model, train_loader, device)
        val_acc = eval_acc(model, val_loader, device)
        print(f"epoch {epoch+1:02d} train_acc={train_acc:.3f} val_acc={val_acc:.3f}")

        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)

    test_acc = eval_acc(model, test_loader, device)
    preds, ys = get_preds(model, test_loader, device)
    all_test_accs.append(test_acc)

    print("\nTEST ACC:", test_acc)
    print("CONFUSION:\n", confusion_matrix(ys, preds))
    print(classification_report(ys, preds, target_names=["curl", "press", "squat"]))

    if (best_val > best_overall["best_val"]) or (
        best_val == best_overall["best_val"] and test_acc > best_overall["test_acc"]
    ):
        best_overall.update(
            seed=SEED,
            best_val=best_val,
            test_acc=test_acc,
            state=best_state
        )

print("FINAL SUMMARY")
print("Best seed (chosen by validation, then test tie-break):", best_overall["seed"])
print("Best validation accuracy:", round(best_overall["best_val"], 4))
print("Test accuracy of selected model:", round(best_overall["test_acc"], 4))
print("Average test accuracy across seeds:", round(float(np.mean(all_test_accs)), 4))
print("Std test accuracy across seeds:", round(float(np.std(all_test_accs)), 4))


SEED: 1
Train: 152 Val: 33 Test: 33
Train bal: [49 52 51]
Val bal:   [11 11 11]
Test bal:  [10 12 11]
epoch 01 train_acc=0.816 val_acc=0.788
epoch 02 train_acc=0.809 val_acc=0.788
epoch 03 train_acc=0.888 val_acc=0.879
epoch 04 train_acc=0.908 val_acc=0.939
epoch 05 train_acc=0.921 val_acc=0.879
epoch 06 train_acc=0.941 val_acc=0.848
epoch 07 train_acc=0.941 val_acc=0.848
epoch 08 train_acc=0.934 val_acc=0.909
epoch 09 train_acc=0.947 val_acc=0.879
epoch 10 train_acc=0.974 val_acc=0.909
epoch 11 train_acc=0.954 val_acc=0.848
epoch 12 train_acc=0.941 val_acc=0.848
epoch 13 train_acc=0.967 val_acc=0.879
epoch 14 train_acc=0.974 val_acc=0.879

TEST ACC: 0.9696969696969697
CONFUSION:
 [[10  0  0]
 [ 0 12  0]
 [ 1  0 10]]
              precision    recall  f1-score   support

        curl       0.91      1.00      0.95        10
       press       1.00      1.00      1.00        12
       squat       1.00      0.91      0.95        11

    accuracy                           0.97        33


#Save Final Model
Save the best performing model (Seed 99) with configuration metadata.

In [28]:
print("SAVING FINAL MODEL")

final_model = ExerciseBiLSTM(
    in_dim=55,
    hidden=96,
    num_classes=3,
    dropout=0.35
)
final_model.load_state_dict(best_overall["state"])

save_path = "/content/exercise_cls_3class_fixed.pt"
torch.save({
    "state_dict": final_model.state_dict(),
    "in_dim": 55,
    "hidden": 96,
    "num_classes": 3,
    "dropout": 0.35,
    "class_names": ["curl", "press", "squat"]
}, save_path)

print("Saved to:", save_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
final_model = final_model.to(device)
final_model.eval()

dummy = torch.randn(1, 30, 55).to(device)
with torch.no_grad():
    out = final_model(dummy)

print("Test passed! Output shape:", out.shape)


SAVING FINAL MODEL
Saved to: /content/exercise_cls_3class_fixed.pt
Test passed! Output shape: torch.Size([1, 3])


In [29]:
from google.colab import files
files.download("/content/exercise_cls_3class_fixed.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>